Filter Eqasim outputs (CSV + optional GPKG)

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Set
import zstandard as zstd
import xml.etree.ElementTree as ET

import pandas as pd
import geopandas as gpd


# ============================================================
# FILTER EQASIM / MATSIM OUTPUTS
# using legacy 17households.csv + 17activities.csv as commune lookup
# ============================================================

# ------------------------------
# CONFIG
# ------------------------------
SIM_OUTPUT_PATH = Path("../simulation_output")   # New output_* files + eqasim_pt.csv + XML.ZST
LEGACY_PATH = Path("../")                         # 17households.csv, 17activities.csv
GPKG_INPUT_PATH = Path("../")                     # optional old gpkg files
OUTPUT_PATH = Path("./")

# 8 peri-urban communes
PERIURBAN_COMMUNES_8: set[int] = {
    17059, 17245, 17109, 17194, 17373, 17315, 17136, 17420
}

# 28 communes of CdA La Rochelle
CDA_COMMUNES_28: set[int] = {
    17010, 17028, 17059, 17094, 17109, 17136, 17142, 17153,
    17190, 17193, 17194, 17200, 17222, 17245, 17264, 17274,
    17291, 17300, 17315, 17373, 17391, 17407, 17413, 17414,
    17420, 17443, 17466, 17483
}

# XML namespaces (for households XML)
MATSim_NS = "http://www.matsim.org/files/dtd"
XSI_NS = "http://www.w3.org/2001/XMLSchema-instance"
ET.register_namespace("", MATSim_NS)
ET.register_namespace("xsi", XSI_NS)


# ------------------------------
# HELPERS
# ------------------------------
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def read_csv_semicolon(path: Path, compression: str | None = None) -> pd.DataFrame:
    return pd.read_csv(path, sep=";", compression=compression)


def normalise_int_series(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def to_str_set(series: pd.Series) -> Set[str]:
    return set(series.dropna().astype(str))


def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[-1]


def write_zstd_text(out_path: Path, text: str) -> None:
    with zstd.open(out_path, "wt", encoding="utf-8") as f:
        f.write(text)


def filter_and_write_csv(df: pd.DataFrame, out_filename: str) -> None:
    out_path = OUTPUT_PATH / out_filename
    df.to_csv(out_path, sep=";", index=False)
    print(f"[OK] Wrote {out_filename} ({len(df)} rows)")


def try_filter_gpkg(
    in_name: str,
    out_name: str,
    key_col: str,
    keep_ids: Set[str],
) -> None:
    """
    Optional filtering of old GPKG files stored in current folder.
    """
    in_path = GPKG_INPUT_PATH / in_name
    if not in_path.exists():
        print(f"[WARN] {in_name} not found -> skip")
        return

    gdf = gpd.read_file(in_path)
    if key_col not in gdf.columns:
        print(f"[WARN] {in_name} does not contain column '{key_col}' -> skip")
        return

    gdf[key_col] = gdf[key_col].astype(str)
    out = gdf[gdf[key_col].isin(keep_ids)].copy()
    out.to_file(OUTPUT_PATH / out_name, driver="GPKG")
    print(f"[OK] Wrote {out_name} ({len(out)} rows)")


def filter_households_xml(
    in_filename: str,
    out_filename: str,
    selected_household_ids: Set[str],
    selected_person_ids: Set[str],
) -> None:
    """
    Filter output_households.xml.zst by household id and selected members.
    """
    in_path = SIM_OUTPUT_PATH / in_filename
    if not in_path.exists():
        print(f"[WARN] {in_filename} not found -> skip")
        return

    with zstd.open(in_path, "rt", encoding="utf-8") as f:
        tree = ET.parse(f)

    root = tree.getroot()

    for child in list(root):
        if strip_ns(child.tag) == "household":
            hid = str(child.get("id"))
            if hid not in selected_household_ids:
                root.remove(child)
                continue

            for members in child:
                if strip_ns(members.tag) != "members":
                    continue
                for member in list(members):
                    if (
                        strip_ns(member.tag) == "personId"
                        and str(member.get("refId")) not in selected_person_ids
                    ):
                        members.remove(member)

    xml_text = '<?xml version="1.0" encoding="UTF-8"?>\n' + ET.tostring(root, encoding="unicode")
    write_zstd_text(OUTPUT_PATH / out_filename, xml_text)
    print(f"[OK] Wrote {out_filename}")


def filter_plans_xml(
    in_filename: str,
    out_filename: str,
    selected_person_ids: Set[str],
) -> Set[str]:
    """
    Filter output_plans.xml.zst by person id.
    Also collect facility ids referenced in kept plans.
    """
    in_path = SIM_OUTPUT_PATH / in_filename
    if not in_path.exists():
        print(f"[WARN] {in_filename} not found -> skip")
        return set()

    with zstd.open(in_path, "rt", encoding="utf-8") as f:
        tree = ET.parse(f)

    root = tree.getroot()
    kept_facility_ids: Set[str] = set()

    for child in list(root):
        if child.tag == "person":
            pid = str(child.get("id"))
            if pid not in selected_person_ids:
                root.remove(child)
            else:
                for act in child.iter("activity"):
                    fid = act.get("facility")
                    if fid:
                        kept_facility_ids.add(str(fid))

    xml_text = (
        '<?xml version="1.0" encoding="utf-8"?>\n'
        '<!DOCTYPE population SYSTEM "http://www.matsim.org/files/dtd/population_v6.dtd">\n'
        + ET.tostring(root, encoding="unicode")
    )
    write_zstd_text(OUTPUT_PATH / out_filename, xml_text)
    print(f"[OK] Wrote {out_filename}")

    return kept_facility_ids


def filter_facilities_xml(
    in_filename: str,
    out_filename: str,
    selected_facility_ids: Set[str],
) -> None:
    """
    Filter output_facilities.xml.zst by facility id.
    Keeps the global <attributes> section.
    """
    in_path = SIM_OUTPUT_PATH / in_filename
    if not in_path.exists():
        print(f"[WARN] {in_filename} not found -> skip")
        return

    with zstd.open(in_path, "rt", encoding="utf-8") as f:
        tree = ET.parse(f)

    root = tree.getroot()

    for child in list(root):
        if child.tag == "facility":
            fid = str(child.get("id"))
            if fid not in selected_facility_ids:
                root.remove(child)

    xml_text = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<!DOCTYPE facilities SYSTEM "http://www.matsim.org/files/dtd/facilities_v2.dtd">\n'
        + ET.tostring(root, encoding="unicode")
    )
    write_zstd_text(OUTPUT_PATH / out_filename, xml_text)
    print(f"[OK] Wrote {out_filename}")


# ------------------------------
# RUN
# ------------------------------
ensure_dir(OUTPUT_PATH)

# ============================================================
# 1) READ INPUT FILES
# ============================================================
# New outputs
persons = read_csv_semicolon(SIM_OUTPUT_PATH / "output_persons.csv.zst", compression="zstd")
activities = read_csv_semicolon(SIM_OUTPUT_PATH / "output_activities.csv.zst", compression="zstd")
trips = read_csv_semicolon(SIM_OUTPUT_PATH / "output_trips.csv.zst", compression="zstd")
pt_legs = read_csv_semicolon(SIM_OUTPUT_PATH / "eqasim_pt.csv.zst", compression="zstd")

# Legacy lookup tables
legacy_households = read_csv_semicolon(LEGACY_PATH / "17households.csv")
legacy_activities = read_csv_semicolon(LEGACY_PATH / "17activities.csv")

# ============================================================
# 2) NORMALISE IDENTIFIERS
# ============================================================
# New outputs
persons["person"] = persons["person"].astype(str)
persons["householdId"] = persons["householdId"].astype(str)

activities["person"] = activities["person"].astype(str)
activities["activity_number"] = normalise_int_series(activities["activity_number"])
activities["activity_id"] = activities["activity_id"].astype(str)
activities["facility_id"] = activities["facility_id"].astype(str)

trips["person"] = trips["person"].astype(str)
trips["trip_number"] = normalise_int_series(trips["trip_number"])
trips["trip_id"] = trips["trip_id"].astype(str)
trips["start_facility_id"] = trips["start_facility_id"].astype(str)
trips["end_facility_id"] = trips["end_facility_id"].astype(str)

pt_legs["person_id"] = pt_legs["person_id"].astype(str)
pt_legs["person_trip_id"] = normalise_int_series(pt_legs["person_trip_id"])

# Legacy lookup
legacy_households["household_id"] = legacy_households["household_id"].astype(str)
legacy_households["commune_id"] = normalise_int_series(legacy_households["commune_id"])

legacy_activities["person_id"] = legacy_activities["person_id"].astype(str)
legacy_activities["activity_index"] = normalise_int_series(legacy_activities["activity_index"])
legacy_activities["commune_id"] = normalise_int_series(legacy_activities["commune_id"])

# output_activities.activity_number starts at 1
# 17activities.activity_index starts at 0
legacy_activities_lookup = legacy_activities[
    ["person_id", "activity_index", "commune_id"]
].copy()
legacy_activities_lookup["activity_number"] = legacy_activities_lookup["activity_index"] + 1

# ============================================================
# 3) STEP 1 = RESIDENTS OF THE 8 PERI-URBAN COMMUNES
# via output_persons.householdId -> 17households.household_id
# ============================================================
persons_with_home_commune = persons.merge(
    legacy_households[["household_id", "commune_id"]],
    left_on="householdId",
    right_on="household_id",
    how="left",
)

home_commune_missing = persons_with_home_commune["commune_id"].isna().sum()
print(f"[INFO] Missing household commune lookup: {home_commune_missing}")

base_person_ids: Set[str] = set(
    persons_with_home_commune.loc[
        persons_with_home_commune["commune_id"].isin(PERIURBAN_COMMUNES_8),
        "person",
    ].astype(str)
)

# ============================================================
# 4) STEP 2 = PERSONS WHOSE ALL ACTIVITIES ARE WITHIN CDA
# via (person, activity_number) -> (person_id, activity_index + 1)
# ============================================================
activities_with_commune = activities.merge(
    legacy_activities_lookup[["person_id", "activity_number", "commune_id"]],
    left_on=["person", "activity_number"],
    right_on=["person_id", "activity_number"],
    how="left",
)

activity_commune_missing = activities_with_commune["commune_id"].isna().sum()
print(f"[INFO] Missing activity commune lookup: {activity_commune_missing}")

in_cda_row = (
    activities_with_commune["commune_id"].isin(CDA_COMMUNES_28)
    & activities_with_commune["commune_id"].notna()
)

all_acts_in_cda = in_cda_row.groupby(activities_with_commune["person"]).all()

cda_person_ids: Set[str] = set(
    all_acts_in_cda[all_acts_in_cda].index.astype(str)
)

# ============================================================
# 5) UNION OF STEP 1 + STEP 2
# ============================================================
selected_person_ids: Set[str] = base_person_ids | cda_person_ids

selected_household_ids: Set[str] = set(
    persons.loc[persons["person"].isin(selected_person_ids), "householdId"].astype(str)
)

print("========== SUMMARY ==========")
print(f"[INFO] Step 1 persons (8 communes): {len(base_person_ids)}")
print(f"[INFO] Step 2 persons (all activities in CdA): {len(cda_person_ids)}")
print(f"[INFO] Total selected persons (union): {len(selected_person_ids)}")
print(f"[INFO] Total selected households: {len(selected_household_ids)}")
print("=============================")

# ============================================================
# 6) FILTER NEW TABULAR OUTPUTS
# ============================================================
persons_out = persons[persons["person"].isin(selected_person_ids)].copy()
activities_out = activities[activities["person"].isin(selected_person_ids)].copy()
trips_out = trips[trips["person"].isin(selected_person_ids)].copy()
pt_legs_out = pt_legs[pt_legs["person_id"].isin(selected_person_ids)].copy()

filter_and_write_csv(persons_out, "output_persons_filtered.csv")
filter_and_write_csv(activities_out, "output_activities_filtered.csv")
filter_and_write_csv(trips_out, "output_trips_filtered.csv")
filter_and_write_csv(pt_legs_out, "eqasim_pt_filtered.csv")

# Diagnostic enriched exports
persons_with_home_commune_out = persons_with_home_commune[
    persons_with_home_commune["person"].isin(selected_person_ids)
].copy()
filter_and_write_csv(
    persons_with_home_commune_out,
    "output_persons_filtered_with_household_commune.csv",
)

activities_with_commune_out = activities_with_commune[
    activities_with_commune["person"].isin(selected_person_ids)
].copy()
filter_and_write_csv(
    activities_with_commune_out,
    "output_activities_filtered_with_commune.csv",
)

# ============================================================
# 7) FILTER XML OUTPUTS
# ============================================================
filter_households_xml(
    in_filename="output_households.xml.zst",
    out_filename="output_households_filtered.xml.zst",
    selected_household_ids=selected_household_ids,
    selected_person_ids=selected_person_ids,
)

plan_facility_ids = filter_plans_xml(
    in_filename="output_plans.xml.zst",
    out_filename="output_plans_filtered.xml.zst",
    selected_person_ids=selected_person_ids,
)

selected_facility_ids: Set[str] = set()
selected_facility_ids |= to_str_set(activities_out["facility_id"])
selected_facility_ids |= to_str_set(trips_out["start_facility_id"])
selected_facility_ids |= to_str_set(trips_out["end_facility_id"])
selected_facility_ids |= plan_facility_ids

filter_facilities_xml(
    in_filename="output_facilities.xml.zst",
    out_filename="output_facilities_filtered.xml.zst",
    selected_facility_ids=selected_facility_ids,
)

# ============================================================
# 8) OPTIONAL: FILTER OLD GPKG FILES STORED IN "."
# ============================================================
try_filter_gpkg("17homes.gpkg", "17homes_filtered.gpkg", "household_id", selected_household_ids)
try_filter_gpkg("17activities.gpkg", "17activities_filtered.gpkg", "person_id", selected_person_ids)
try_filter_gpkg("17trips.gpkg", "17trips_filtered.gpkg", "person_id", selected_person_ids)
try_filter_gpkg("17commutes.gpkg", "17commutes_filtered.gpkg", "person_id", selected_person_ids)

print("[OK] Filtering completed.")

[INFO] Missing household commune lookup: 14


[INFO] Missing activity commune lookup: 1


========== SUMMARY ==========
[INFO] Step 1 persons (8 communes): 1647
[INFO] Step 2 persons (all activities in CdA): 15573
[INFO] Total selected persons (union): 16046
[INFO] Total selected households: 9591


[OK] Wrote output_persons_filtered.csv (16046 rows)


[OK] Wrote output_activities_filtered.csv (68346 rows)


...

[OK] Wrote output_facilities_filtered.xml.zst


[OK] Wrote 17homes_filtered.gpkg (9586 rows)


[OK] Wrote 17activities_filtered.gpkg (70469 rows)


[OK] Wrote 17trips_filtered.gpkg (54423 rows)


[OK] Wrote 17commutes_filtered.gpkg (4662 rows)
[OK] Filtering completed.
